# Lab 10: Machine Learning and Chemistry
*Elements of Data Science* <br>
Welcome to Lab 10! In Lab 8 you predicted an alcohol's boiling point with a **straight line**. Today you will build a different kind of predictor that makes no assumption about the shape of the relationship: **k-nearest neighbors (kNN)**. The idea is simple — *molecules that are alike tend to behave alike* — and it is one of the foundations of machine learning.

1. **Distance** — measure how alike two data points are.
2. **Training and test sets** — learn from some data, judge the predictions on data the model has never seen.
3. **Predict** alcohol boiling points with kNN and compare with your Lab 8 regression line.
4. **Scale** the features so that no single feature dominates the distance.
5. **Choose k** fairly, using a **validation set**.
6. **Apply** it all to a real research problem: predicting molecular **acidity (pKa)**, and classifying molecules as stronger or weaker acids.

The acidity data were computed by [Prof. Vince Voelz](http://www.voelzlab.org) of the Temple Chemistry department and graduate student Robert Raddi ([Stacking Gaussian processes to improve pKa predictions in the SAMPL7 challenge](https://link.springer.com/article/10.1007/s10822-021-00411-8)).

This material is covered in [Chapter 17](https://inferentialthinking.com/chapters/17/Classification.html) of the online <i>Inferential Thinking</i> textbook.

<br>**<center>Learning Goals**

|Area|Concept|
|---|---|
|Features and target|The **features** are what we predict *from*; the **target** is what we predict|
|Euclidean distance|$\sqrt{\sum (\text{difference in each feature})^2}$ measures how alike two rows are|
|k-nearest neighbors|Predict with the average (regression) or majority vote (classification) of the $k$ most similar training rows|
|Training / validation / test|Learn from training data, make choices with validation data, report on test data once|
|RMSE|Root mean squared error: the typical size of a prediction error|
|Scaling|Convert features to standard units so each one counts fairly in the distance|

## ⚙️ Initialization — DO NOT EDIT
The next two cells load the course tools (auto-checks, quizzes, styling) and record the lab version. **Do not change them** — with one exception:

👁️ **Accessible view (optional):** set `ACCESSIBLE_VIEW = True` in the cell below for larger, more readable text and colorblind-safe plots. You can switch back any time by setting it to `False` and re-running the cell (then *Kernel → Restart* to clear the text styling).

👉 **Run the cell below first** (Shift+Enter) before anything else in this lab.

In [ ]:
# 🔧 Run this cell first! (Shift+Enter)
ACCESSIBLE_VIEW = False     # True = larger text and colorblind-safe plots

import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "shared-public")))
if ACCESSIBLE_VIEW:
    from init_access import *
else:
    from init import *

In [ ]:
# 🔖 Lab version — do not edit. Used by the automated checks below.
ver = '2026V110'

### 🧰 Standard imports — start your own notebooks with these
The `init` cell above loads the course tools. The four lines below are the **only** imports you need for your own data science notebooks — copy them into the first cell of any new notebook:

```python
import numpy as np                 # arrays, np.arange, np.random.choice, np.mean, np.std ...
from datascience import *          # Table, make_array, are ...
%matplotlib inline
import matplotlib.pyplot as plt    # plotting
```
Run them here too so every name is defined:

In [ ]:
# Standard imports for this lab
import numpy as np
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plt

Now run the cell below to enter your name.

In [ ]:
# Enter your name as a string
name = ...

## 1. The Nearest-Neighbor Idea

Suppose you want to predict something about a new molecule. A natural strategy: find the molecules you already know about that are **most similar** to it, and assume the new one behaves like them. That is the **nearest-neighbor** idea ([Ch. 17.1](https://inferentialthinking.com/chapters/17/1/Nearest_Neighbors.html)).

To make "most similar" precise, we describe each molecule by numbers called **features** (molecular weight, number of carbons, ...) and measure the **distance** between them. With two features, $x$ and $y$, this is the familiar straight-line distance from geometry:

$$ d = \sqrt{(x_1 - x_2)^2 + (y_1 - y_2)^2} $$

With more features, we keep adding squared differences — one for each feature ([Ch. 17.3](https://inferentialthinking.com/chapters/17/3/Rows_of_Tables.html)):

$$ d = \sqrt{\sum_{\text{features}} (\text{difference})^2} $$

The cell below plots seven **training** points (red) whose values we know, and one new **test** point (blue). Which training point is nearest?

In [ ]:
x = make_array(3, 8, 5, 6, 1, 9, 8)
y = make_array(4, 5, 7, 4, 6, 9, 4)
point_number = np.arange(1, 8)
test_point = make_array(4, 8)

plt.figure(figsize=(6, 5))
plt.scatter(x, y, color='red', s=40, label='training points')
plt.scatter(test_point.item(0), test_point.item(1), color='blue', s=120, label='test point')
for i in np.arange(len(x)):
    plt.annotate(point_number.item(i), (x.item(i) + 0.15, y.item(i) + 0.15))
plt.xlabel('feature x'); plt.ylabel('feature y'); plt.legend()
plt.show()

### <font color=blue> **Question 1.1.** </font>
Complete the function `distance`, which returns the Euclidean distance between two points given as arrays of the same length. It should work for **any** number of features.

*Hint:* subtract the arrays, square, add up with `np.sum`, then take the square root with `np.sqrt`.

In [ ]:
def distance(pt1, pt2):
    "The Euclidean distance between two points, given as arrays."
    return ...

# Try it: the distance between (0, 0) and (3, 4) should be 5
distance(make_array(0, 0), make_array(3, 4))

In [ ]:
check('tests/q1.1.py')

Now use your function to find the distance from the blue test point to each red training point, and sort from nearest to farthest.

In [ ]:
toy_distances = make_array()
for i in np.arange(len(x)):
    training_point = make_array(x.item(i), y.item(i))
    toy_distances = np.append(toy_distances, distance(training_point, test_point))

toy = Table().with_columns('training point', point_number, 'x', x, 'y', y, 'distance', toy_distances)
toy.sort('distance')

### <font color=blue> **Question 1.2.** </font>
1. Which training point is the **nearest** neighbor of the test point? Assign its number to `nearest_point`.
2. Using `toy`, make an array of the numbers of the **3 nearest** neighbors (nearest first) and assign it to `three_nearest`. *Hint:* sort, `.take(np.arange(3))`, then `.column(...)`.

In [ ]:
nearest_point = ...
three_nearest = ...
three_nearest

In [ ]:
check('tests/q1.2.py')

## 2. Predicting Boiling Points with k-Nearest Neighbors

Back to the alcohols from Lab 8. `ROH_data.csv` has 72 alcohols with their boiling point `bp` (K), molecular weight `MW` (g/mol), number of `carbons`, `degree` (1 = primary, 2 = secondary, 3 = tertiary), and whether they contain a ring (`cyclic`: 1 = yes).

In machine learning language:
- the **target** is what we want to predict: `bp`
- the **features** are what we predict it *from*: we start with `MW`, just like the regression line in Lab 8

We store them as `roh_features` and `roh_target` (*ROH* is chemists' shorthand for an alcohol), so they won't get mixed up with the acidity features in Part 3.

In [ ]:
ROH_data = Table.read_table('ROH_data.csv')
ROH_data

In [ ]:
roh_features = make_array('MW')
roh_target = 'bp'

### Training and test sets
A model that has seen the answers can't be graded fairly on them. So before doing anything else, we **randomly** split the data into two parts ([Ch. 17.2](https://inferentialthinking.com/chapters/17/2/Training_and_Testing.html)):

- the **training set**: the molecules whose boiling points the model is allowed to use — these are the "neighbors"
- the **test set**: molecules we hide from the model, then use at the end to measure how good its predictions are

The Table method `.split(n)` shuffles the rows and returns two Tables: the first `n` rows and the rest ([documentation](https://datascience.readthedocs.io/en/master/_autosummary/datascience.tables.Table.split.html)).

> 🎲 **Why `np.random.seed`?** A seed makes the "random" shuffle the same every time you run the cell, so your results match your classmates' and the auto-checks. Change the seed and you get a different (equally valid) split.

### <font color=blue> **Question 2.1.** </font>
Split `ROH_data` into `train_roh` with **80%** of the rows and `test_roh` with the rest. `.split` needs a whole number, so use `int(...)`.

In [ ]:
np.random.seed(1013)
train_roh, test_roh = ROH_data.split(...)
print(train_roh.num_rows, 'training and', test_roh.num_rows, 'test molecules')

In [ ]:
check('tests/q2.1.py')

### The k-nearest-neighbors toolbox
The functions below turn the idea from Part 1 into a predictor. **Read them carefully** — you'll use them for the rest of the lab.

- `distances` adds a column with the distance from every training row to one `example` row, using only the feature columns.
- `closest` keeps the `k` training rows with the smallest distances.
- `predict_knn` predicts the target for `example` as the **average** target of those `k` neighbors ([Ch. 17.6](https://inferentialthinking.com/chapters/17/6/Multiple_Regression.html)).
- `predict_all` makes a prediction for every row of a Table.

In [ ]:
def distances(training, example, features):
    "Training Table with a 'Distance' column: distance from each training row to example."
    dists = make_array()
    for row in training.select(features).rows:
        dists = np.append(dists, distance(np.array(row), np.array(example)))
    return training.with_columns('Distance', dists)

def closest(training, example, k, features):
    "The k rows of training that are nearest to example."
    return distances(training, example, features).sort('Distance').take(np.arange(k))

def predict_knn(training, example, k, features, target):
    "Predict the target for example: the average target of its k nearest neighbors."
    return np.mean(closest(training, example, k, features).column(target))

def predict_all(training, evaluate, k, features, target):
    "An array of kNN predictions, one for every row of the Table evaluate."
    predictions = make_array()
    for example in evaluate.select(features).rows:
        predictions = np.append(predictions, predict_knn(training, example, k, features, target))
    return predictions

Let's try it on the first molecule in the test set. Here are its features and its 3 nearest neighbors in the training set:

In [ ]:
example = test_roh.select(roh_features).row(0)
print('Test molecule features:', example)
print('Actual boiling point:  ', test_roh.column(roh_target).item(0), 'K')
closest(train_roh, example, 3, roh_features)

### <font color=blue> **Question 2.2.** </font>
Use `predict_knn` with **k = 3** to predict the boiling point of this test molecule, and store it as `bp_prediction`. It should equal the average `bp` of the three neighbors above.

In [ ]:
bp_prediction = ...
bp_prediction

In [ ]:
check('tests/q2.2.py')

### How good are the predictions overall? RMSE
One prediction tells us little. To judge the model we predict **every** test molecule and summarize the errors with the **root mean squared error (RMSE)**:

$$ \text{RMSE} = \sqrt{\text{mean of } (\text{actual} - \text{predicted})^2} $$

It is the same "square the errors so they can't cancel" idea as the sum of squared errors in Lab 8, but averaged and square-rooted so it has the **same units as the target** (here, kelvin). Roughly, it is the size of a typical prediction error ([Ch. 15.3](https://inferentialthinking.com/chapters/15/3/Method_of_Least_Squares.html)).

### <font color=blue> **Question 2.3.** </font>
Complete `rmse_knn` by filling in `errors` (actual minus predicted). Then compute the test-set RMSE with **k = 3** and store it as `knn_rmse`.

In [ ]:
def rmse_knn(training, evaluate, k, features, target):
    "Root mean squared error of kNN predictions for every row of evaluate."
    predictions = predict_all(training, evaluate, k, features, target)
    errors = ...
    return np.sqrt(np.mean(errors ** 2))

knn_rmse = ...
print(f'kNN (k = 3) test RMSE: {knn_rmse:.1f} K')

In [ ]:
check('tests/q2.3.py')

### Compare with the regression line from Lab 8
Is kNN any better than a straight line? To compare fairly, the line must also be fit **using only the training set**, then judged on the same test set. Here is your regression toolbox from Lab 8:

In [ ]:
# Your regression toolbox from Lab 8 (Question 4.4)
def standard_units(any_numbers):
    "Convert an array of numbers to standard units."
    return (any_numbers - np.mean(any_numbers)) / np.std(any_numbers)

def correlation(t, label_x, label_y):
    "The correlation coefficient r between two columns of Table t."
    return np.mean(standard_units(t.column(label_x)) * standard_units(t.column(label_y)))

def slope(t, label_x, label_y):
    "Slope of the regression line predicting label_y from label_x, in original units."
    r = correlation(t, label_x, label_y)
    return r * np.std(t.column(label_y)) / np.std(t.column(label_x))

def intercept(t, label_x, label_y):
    "Intercept of the regression line predicting label_y from label_x, in original units."
    return np.mean(t.column(label_y)) - slope(t, label_x, label_y) * np.mean(t.column(label_x))

### <font color=blue> **Question 2.4.** </font>
Fit the regression line to the **training** set (`train_roh`), then use it to predict the boiling points of the **test** set. The RMSE line is done for you.

In [ ]:
roh_slope = ...
roh_intercept = ...
line_predictions = ...
line_rmse = np.sqrt(np.mean((test_roh.column(roh_target) - line_predictions) ** 2))

print(f'Regression line test RMSE: {line_rmse:.1f} K')
print(f'kNN (k = 3) test RMSE:     {knn_rmse:.1f} K')

In [ ]:
check('tests/q2.4.py')

Run the cell below to see both sets of predictions for the test molecules.

In [ ]:
knn_predictions = predict_all(train_roh, test_roh, 3, roh_features, roh_target)
mw = test_roh.column('MW')

plt.figure(figsize=(8, 5))
plt.scatter(train_roh.column('MW'), train_roh.column(roh_target), s=15, alpha=0.3, color='gray', label='training molecules')
plt.scatter(mw, test_roh.column(roh_target), s=50, label='test molecules (actual)')
plt.scatter(mw, knn_predictions, s=50, marker='x', label='kNN prediction (k = 3)')
line_x = make_array(30, 190)
plt.plot(line_x, roh_slope * line_x + roh_intercept, color='black', lw=2, label='regression line (training fit)')
plt.xlabel('MW [g/mol]'); plt.ylabel('bp [K]'); plt.legend()
plt.show()

### <font color=blue> **Question 2.4 Discussion.** </font>
<font color=green>
1. Which method had the smaller test RMSE? With only about 15 test molecules, would you expect the winner to be the same for a different random split?
2. Look at the kNN predictions (✕). Why do molecules with the **same** MW always get the **same** kNN prediction, even when their actual boiling points differ? What extra feature from Lab 8 might help tell them apart?
3. You might be tempted to try several values of k (or different features) and keep whichever gives the lowest **test** RMSE. Why would that make the test set a less honest judge? (Part 3 shows the fix.)
</font>

---

ANSWER

---

#### Save notebook before check

In [ ]:
check('tests/q2.4d.py')

### 📝 Quick Check — Nearest Neighbors
Test what you've learned about distance, kNN, and training/test sets. This is just for practice — answer as many times as you like, it isn't recorded.

In [ ]:
with open("questions10A.json", "r") as f:
    display_quiz(json.load(f), shuffle_answers=True)

## 3. Predicting Molecular Acidity (pKa)

Now a real research problem. An **acid** is a molecule that can give up a proton (H⁺) in water. How readily it does so is measured by its **pKa**:

- **lower pKa → stronger acid** (gives up its proton more easily)
- at a pH equal to its pKa, a molecule is half protonated and half deprotonated
- at pH 7 (neutral water, close to the pH of blood), a molecule with **pKa below 7** has mostly given up its proton

pKa matters enormously in medicine: whether a drug carries a charge at the pH of the body affects how it is absorbed and where it goes. Measuring pKa takes lab time, so chemists want to **predict** it from the structure. See [OpenStax Chemistry 2e, Ch. 14](https://openstax.org/books/chemistry-2e/pages/14-introduction) for more on acids and bases.

<center><img src='acid_base_pKa.png' width=900 alt='Acid-base equilibria of glycine with the pKa of each step'></center>

Each row of `pKa_small.csv` is one acidic site in one molecule. Along with the measured `pKa`, Prof. Voelz's group **computed** features that should be related to acidity ([Raddi & Voelz 2021](https://link.springer.com/article/10.1007/s10822-021-00411-8)):

|Feature|Meaning|
|---|---|
|`Weight`|molecular weight (g/mol)|
|`∆H`|change in enthalpy (kJ/mol) between the protonated and deprotonated forms|
|`∆G_solv`|change in solvation free energy (kJ/mol): how differently water stabilizes the two forms|
|`Bond Order`|strength of the bond to the atom holding the acidic proton|
|`SASA`|solvent-accessible surface area around the acidic site: how exposed it is to water|

In [ ]:
pka_data = Table.read_table('pKa_small.csv')
pka_data

Many of these molecules are medicines. RDKit (from Lab 8) draws them from their SMILES strings:

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw

drugs = make_array('celecoxib', 'Sertraline', 'Losartan')
uses = make_array('anti-inflammatory (NSAID)', 'antidepressant (SSRI)', 'blood pressure')
drug_molecules = []
drug_labels = []
for i in np.arange(len(drugs)):
    row = pka_data.where('Name', drugs.item(i))
    drug_molecules.append(Chem.MolFromSmiles(row.column('smiles').item(0)))
    drug_labels.append(f"{drugs.item(i)}: {uses.item(i)}\npKa = {row.column('pKa').item(0)}")

options = Draw.MolDrawOptions()
options.legendFontSize = 18
options.legendFraction = 0.22
Draw.MolsToGridImage(drug_molecules, molsPerRow=3, subImgSize=(330, 260), legends=drug_labels, drawOptions=options)

Run the cell below a few times to meet some random molecules from the data set.

In [ ]:
some = pka_data.sample(4, with_replacement=False)
Draw.MolsToGridImage([Chem.MolFromSmiles(s) for s in some.column('smiles')], molsPerRow=4, subImgSize=(250, 220),
                     legends=[f'{n}\npKa = {p}' for n, p in zip(some.column('Name'), some.column('pKa'))],
                     drawOptions=options)

### 3.1 Training, validation, and test sets
In Part 2 we noticed a problem: if we use the test set to *choose* things (like k), it is no longer an honest judge. The standard fix is a **third** set ([Ch. 17.2](https://inferentialthinking.com/chapters/17/2/Training_and_Testing.html)):

|Set|Used for|Analogy|
|---|---|---|
|**Training**|the neighbors that make the predictions|studying the textbook examples|
|**Validation**|comparing choices: which k, which features, scaled or not|practice exams — take as many as you like|
|**Test**|one final, honest measure of the chosen model|the final exam — taken **once**|

We use a 60% / 20% / 20% split.

### <font color=blue> **Question 3.1.** </font>
Split `pka_data` into `train` (60% of the rows) and `rest`. Then split `rest` in half into `validation` and `test`. Remember `int(...)`.

In [ ]:
np.random.seed(1013)
train, rest = pka_data.split(...)
validation, test = rest.split(...)
print(train.num_rows, 'training,', validation.num_rows, 'validation,', test.num_rows, 'test')

In [ ]:
check('tests/q3.1.py')

### 3.2 Why scaling matters
Look at the spread (SD) of each feature in the training set:

In [ ]:
pka_features = make_array('Weight', '∆H', '∆G_solv', 'Bond Order', 'SASA')
pka_target = 'pKa'

spreads = make_array()
for label in pka_features:
    spreads = np.append(spreads, np.std(train.column(label)))
Table().with_columns('feature', pka_features, 'SD in training set', spreads)

The features are measured in completely different units, and their spreads differ by factors of a thousand or more. In the distance formula, a difference of 0.05 in `Bond Order` (a big change for a bond!) adds almost nothing, while the feature with the largest SD swamps everything else. **The distance mostly ignores the small-number features.**

The fix is the ruler from Lab 8: convert every feature to **standard units**, so each one is measured in "SDs from average" and counts on equal terms.

One subtle rule: we measure with the **training set's** mean and SD, even when converting the validation and test sets. Otherwise information from the held-out sets would leak into the model — like peeking at the final exam to decide how to study.

In [ ]:
def standardize(t, features, ruler):
    "Copy of Table t with each feature in standard units, using the mean and SD of `ruler` (the training set)."
    result = t
    for label in features:
        mean = np.mean(ruler.column(label))
        sd = np.std(ruler.column(label))
        result = result.with_columns(label, (t.column(label) - mean) / sd)
    return result

### <font color=blue> **Question 3.2.** </font>
Convert `train`, `validation`, and `test` to standard units, always using `train` as the ruler. Then run the cell to compare the validation RMSE (k = 5) **before** and **after** scaling, along with a "no-skill" baseline that ignores the features and always guesses the average training pKa.

In [ ]:
train_su = standardize(train, pka_features, train)
validation_su = ...
test_su = ...

raw_rmse = rmse_knn(train, validation, 5, pka_features, pka_target)
scaled_rmse = rmse_knn(train_su, validation_su, 5, pka_features, pka_target)
guess_rmse = np.sqrt(np.mean((validation.column(pka_target) - np.mean(train.column(pka_target))) ** 2))

print(f'Always guess the average pKa:  validation RMSE = {guess_rmse:.2f}')
print(f'kNN, raw features:             validation RMSE = {raw_rmse:.2f}')
print(f'kNN, standardized features:    validation RMSE = {scaled_rmse:.2f}')

In [ ]:
check('tests/q3.2.py')

### 3.3 Choosing k with the validation set
The number of neighbors, k, is a choice we make *before* the model sees any data — a **hyperparameter**. Too small and a prediction depends on one or two possibly unusual neighbors; too large and it averages over molecules that aren't really similar. Let the validation set decide. (This cell takes a little while to run.)

In [ ]:
k_values = make_array(1, 3, 5, 7, 9, 15, 25)
validation_rmse = make_array()
for k in k_values:
    validation_rmse = np.append(validation_rmse, rmse_knn(train_su, validation_su, k, pka_features, pka_target))

k_results = Table().with_columns('k', k_values, 'validation RMSE', validation_rmse)
k_results.plot('k', marker='o')
plt.ylabel('validation RMSE (pKa units)')
plt.show()
k_results

### <font color=blue> **Question 3.3.** </font>
Assign `best_k` to the value of k with the **smallest validation RMSE**. Then run the cell to use that k **once** on the test set — the final exam.

In [ ]:
best_k = ...

test_predictions = predict_all(train_su, test_su, best_k, pka_features, pka_target)
test_errors = test_su.column(pka_target) - test_predictions
test_rmse = np.sqrt(np.mean(test_errors ** 2))
test_guess_rmse = np.sqrt(np.mean((test_su.column(pka_target) - np.mean(train_su.column(pka_target))) ** 2))

plt.hist(test_errors, bins=np.arange(-10, 10.5, 1), edgecolor='black')
plt.xlabel('actual pKa − predicted pKa')
plt.ylabel('number of test molecules')
plt.title(f'kNN with k = {best_k}: test RMSE = {test_rmse:.2f}')
plt.show()
print(f'Test RMSE, kNN (k = {best_k}): {test_rmse:.2f}    Test RMSE, always guess the average: {test_guess_rmse:.2f}')

In [ ]:
check('tests/q3.3.py')

### <font color=blue> **Question 3.3 Discussion.** </font>
<font color=green>
1. Did scaling change the validation RMSE much? Using the table of SDs, explain which features the raw distance was mostly ignoring. If scaling barely helped, what might that tell you about those features?
2. How does the kNN test RMSE compare with simply guessing the average pKa? Is kNN a big improvement or a modest one? Why might predicting pKa from these five numbers be hard?
3. Why do we look at the test set only **once**, after choosing k?
</font>

---

ANSWER

---

#### Save notebook before check

In [ ]:
check('tests/q3.3d.py')

### 3.4 Classification: stronger or weaker acid?
Sometimes a chemist doesn't need the exact pKa, just a category: **will this molecule have given up its proton at pH 7?** That is a **classification** problem — the target is a category, not a number ([Ch. 17.4](https://inferentialthinking.com/chapters/17/4/Implementing_the_Classifier.html)).

The only change to kNN: instead of averaging the neighbors' values, the k neighbors **vote**, and the most common class wins. Below we label each molecule `'stronger'` (pKa < 7) or `'weaker'` (pKa ≥ 7) and build the classifier.

In [ ]:
def acid_class(pka_values):
    "Label each pKa as a 'stronger' (pKa < 7) or 'weaker' (pKa >= 7) acid."
    return np.where(pka_values < 7, 'stronger', 'weaker')

train_su = train_su.with_columns('class', acid_class(train_su.column('pKa')))
test_su = test_su.with_columns('class', acid_class(test_su.column('pKa')))

def classify_knn(training, example, k, features):
    "The most common class among the k nearest neighbors of example."
    votes = closest(training, example, k, features).group('class').sort('count', descending=True)
    return votes.column('class').item(0)

class_predictions = make_array()
for example in test_su.select(pka_features).rows:
    class_predictions = np.append(class_predictions, classify_knn(train_su, example, best_k, pka_features))

test_su.select('Name', 'pKa', 'class').with_columns('predicted class', class_predictions).show(8)

### <font color=blue> **Question 3.4.** </font>
The **accuracy** of a classifier is the proportion of test molecules it classifies correctly ([Ch. 17.5](https://inferentialthinking.com/chapters/17/5/Accuracy_of_the_Classifier.html)). Compute it and store it as `accuracy`.

*Hint:* comparing two arrays with `==` gives an array of `True`/`False`, and `np.mean` of that array is the proportion that are `True`.

The cell also computes a baseline: the accuracy of always predicting whichever class is most common in the training set.

In [ ]:
accuracy = ...

most_common = train_su.group('class').sort('count', descending=True).column('class').item(0)
baseline_accuracy = np.mean(test_su.column('class') == most_common)
print(f'kNN accuracy:                            {accuracy:.2f}')
print(f"Always predict '{most_common}' accuracy:   {baseline_accuracy:.2f}")

In [ ]:
check('tests/q3.4.py')

<font color=green>**Think about it:** Is the classifier's accuracy impressive compared with the baseline? Which mistakes would matter more to a chemist designing a drug — calling a stronger acid weaker, or the reverse?</font>

### 📝 Quick Check — Scaling, Validation, and Classification
This is just for practice — answer as many times as you like, it isn't recorded.

In [ ]:
with open("questions10B.json", "r") as f:
    display_quiz(json.load(f), shuffle_answers=True)

## 📝 Quick Review Quiz
Unlike the quick checks above, this quiz **is recorded** — you get two attempts per question, so take your time.

In [ ]:
# Interactive quiz — run this cell and answer the questions!
from stemds_quiz import open_quiz
quiz_result = open_quiz("questions10C.json", name, user, quiz_id="Lab10C", max_attempts=2)

## 4. Reflection

### <font color=blue> **Question 4.** </font>
At the end of each lab, please include a reflection.
* How did this lab go?
* Can you think of another problem — in science or in everyday life — where k-nearest neighbors could make predictions?
* Were there questions you found especially challenging that you would like your instructor to review in class?
* How long did the lab take you to complete?

Share your feedback so we can continue to improve this class!

**Use the markdown cell below this one and write your reflection on this lab.**

---

ANSWER

---

#### Save notebook before check

In [ ]:
check('tests/q4d.py')

## 🏁 Final Step — Check, Submit to Canvas, Then Submit Lab

Follow these three steps **in order**:

1. ✅ **Run the cell below** to test all your answers and see your score.
2. 📤 **Upload to Canvas.** Download your notebook as a `.ipynb` file ([instructions](https://laserchemist.github.io/data/submission-instructions.html#ipynb)) and export an HTML copy ([instructions](https://laserchemist.github.io/data/submission-instructions.html#html)), then upload that HTML file to this lab's Canvas assignment.
3. 🚀 **Only then run the final cell and click Submit Lab.** This records your score to the course gradebook — treat it as confirmation that steps 1 and 2 are already done.

In [ ]:
# Run all tests at once
from gofer.ok import check
correct = 0
questions = ['1.1', '1.2',
             '2.1', '2.2', '2.3', '2.4', '2.4d',
             '3.1', '3.2', '3.3', '3.3d', '3.4',
             '4d']
for x in questions:
    print("Testing question {}: ".format(x))
    display(check("tests/q{}.py".format(x)))
    score = check('tests/q{}.py'.format(str(x)))
    if score.grade == 1.0:
        correct += 1

In [ ]:
from lab_submit import submit_lab
submit_lab(name, user, correct, questions, lab_id="Lab10")